# Validate against 2016-2017 baseline

Use the monthly historical files as a reinforcement and validation layer for the hotspot proxy.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.append(str(PROJECT_ROOT / 'src'))

from clean_train import clean_train_dataframe
from hotspot_features import add_flow_features, add_time_features, aggregate_station_hotspots


In [ ]:
folder = PROJECT_ROOT / 'data' / 'raw' / 'train_2016_2017_monthly'
frames = []
for path in sorted(folder.glob('*.csv')):
    frame = pd.read_csv(path)
    frame['source_file'] = path.name
    frames.append(frame)

df_raw_2016 = pd.concat(frames, ignore_index=True)
df_clean_2016, metadata_2016 = clean_train_dataframe(df_raw_2016)
print(metadata_2016)
df_clean_2016.head()

In [ ]:
df_flow_2016 = add_flow_features(df_clean_2016, trip_col='trip_id', seq_col='stop_sequence', occupancy_col='occupancy_numeric')
df_flow_2016 = add_time_features(df_flow_2016, time_col='event_time')
df_hotspots_2016 = aggregate_station_hotspots(
    df_flow_2016,
    station_col='station_name',
    lat_col='latitude' if 'latitude' in df_flow_2016.columns else None,
    lon_col='longitude' if 'longitude' in df_flow_2016.columns else None,
)
df_hotspots_2016.head()